In [1]:
# EDA de Matriculados UNSTA

Este notebook realiza un analisis exploratorio de los snapshots exportados a MinIO:
- `vw_dm_matricula_total`
- `vw_dm_matricula_carrera`

Objetivo:
- Entender estructura y calidad de datos.
- Revisar patrones temporales (anio/cuatrimestre).
- Preparar insumos para comparacion de modelos baseline vs series temporales.

SyntaxError: invalid syntax (4275163544.py, line 3)

In [1]:
# Descripcion:
# Esta celda importa librerias, configura estilo visual y define variables de entorno
# para conectarse a MinIO/S3 y al bucket donde estan los snapshots.
import os
import re
from io import BytesIO

import boto3
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

# Configuracion MinIO / S3 compatible
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minio")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minio123")
os.environ.setdefault("AWS_ENDPOINT_URL_S3", "http://localhost:9000")

BUCKET = os.getenv("DATA_REPO_BUCKET_NAME", "data")
BASE_PREFIX = "mysql_exports/unsta"

print("AWS_ENDPOINT_URL_S3:", os.environ.get("AWS_ENDPOINT_URL_S3"))
print("BUCKET:", BUCKET)

ModuleNotFoundError: No module named 'boto3'

In [5]:
# Descripcion:
# Esta celda define funciones auxiliares para listar snapshots, elegir el mas reciente,
# leer CSV desde MinIO y cargar los dos datasets principales (total y carrera).
SNAPSHOT_REGEX = re.compile(r"snapshot_(\d{8}T\d{6})\.csv$")


def get_s3_client():
    return boto3.client(
        "s3",
        endpoint_url=os.getenv("AWS_ENDPOINT_URL_S3"),
        aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
        aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
    )


def list_snapshot_keys(s3_client, bucket: str, prefix: str):
    keys = []
    token = None
    while True:
        kwargs = {"Bucket": bucket, "Prefix": prefix}
        if token:
            kwargs["ContinuationToken"] = token
        response = s3_client.list_objects_v2(**kwargs)
        for obj in response.get("Contents", []):
            key = obj["Key"]
            if SNAPSHOT_REGEX.search(key):
                keys.append(key)
        if not response.get("IsTruncated"):
            break
        token = response.get("NextContinuationToken")
    return sorted(keys)


def latest_snapshot_key(keys):
    if not keys:
        raise ValueError("No se encontraron snapshots para el prefijo indicado.")
    return sorted(keys, key=lambda k: SNAPSHOT_REGEX.search(k).group(1))[-1]


def read_csv_from_s3(s3_client, bucket: str, key: str) -> pd.DataFrame:
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(BytesIO(obj["Body"].read()))


s3 = get_s3_client()

key_total = latest_snapshot_key(
    list_snapshot_keys(s3, BUCKET, f"{BASE_PREFIX}/vw_dm_matricula_total/")
)
key_carrera = latest_snapshot_key(
    list_snapshot_keys(s3, BUCKET, f"{BASE_PREFIX}/vw_dm_matricula_carrera/")
)

print("Snapshot total:", key_total)
print("Snapshot carrera:", key_carrera)

df_total = read_csv_from_s3(s3, BUCKET, key_total)
df_carrera = read_csv_from_s3(s3, BUCKET, key_carrera)

df_total.columns = [c.strip().lower() for c in df_total.columns]
df_carrera.columns = [c.strip().lower() for c in df_carrera.columns]

print("df_total shape:", df_total.shape)
print("df_carrera shape:", df_carrera.shape)


2024/02/12 22:20:13 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2024/02/12 22:20:13 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'c63a5b84f7ec4df88d52f598f9468570', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2024/02/12 22:20:15 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/facundolucianna/Investigacion/env/notebook_env_python3_11/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils."


In [3]:
# Descripcion:
# Esta celda muestra una primera inspeccion: nombres de columnas y primeras filas
# para entender la estructura general de ambos datasets.
# Vista rapida de datos
print("Columnas total:", list(df_total.columns))
print("Columnas carrera:", list(df_carrera.columns))

display(df_total.head())
display(df_carrera.head())

NameError: name 'df_total' is not defined

In [ ]:
# Descripcion:
# Esta celda evalua calidad de datos en cada tabla: cantidad de nulos,
# filas duplicadas y dimension general.
# Calidad de datos: nulos y duplicados

def data_quality_report(df: pd.DataFrame, name: str):
    nulls = df.isna().sum().sort_values(ascending=False)
    nulls = nulls[nulls > 0]
    print(f"\n===== {name} =====")
    print("Filas:", len(df), "| Columnas:", len(df.columns))
    print("Duplicados exactos:", df.duplicated().sum())
    if len(nulls) == 0:
        print("Sin nulos.")
    else:
        display(nulls.to_frame("null_count"))


data_quality_report(df_total, "TOTAL")
data_quality_report(df_carrera, "CARRERA")

In [ ]:
# Descripcion:
# Esta celda construye utilidades para:
# 1) inferir la columna objetivo de matriculados,
# 2) crear el eje temporal (anio + cuatrimestre),
# 3) dejar listas las variables para analisis/modelado.
# Utilidades para preparar eje temporal y detectar columna objetivo

def infer_target_col(df: pd.DataFrame, excluded: set[str]):
    preferred = [
        "matriculados", "total_matriculados", "cantidad_matriculados", "n_matriculados", "cantidad", "total"
    ]
    for c in preferred:
        if c in df.columns and c not in excluded:
            return c

    candidates = [
        c for c in df.columns
        if c not in excluded and pd.api.types.is_numeric_dtype(df[c])
    ]
    if not candidates:
        raise ValueError("No se pudo inferir columna objetivo.")
    return candidates[0]


def build_time_order(df: pd.DataFrame):
    year_col = next((c for c in ["periodo", "anio", "year"] if c in df.columns), None)
    term_col = next((c for c in ["id_cuatrimestre", "cuatrimestre", "term", "semester"] if c in df.columns), None)
    if year_col is None:
        raise ValueError("No se encontro columna de anio (periodo/anio/year).")

    out = df.copy()
    out[year_col] = pd.to_numeric(out[year_col], errors="coerce")
    out = out.dropna(subset=[year_col])
    out[year_col] = out[year_col].astype(int)

    if term_col is not None:
        out[term_col] = pd.to_numeric(out[term_col], errors="coerce").fillna(0).astype(int)
        out["time_order"] = out[year_col] * 10 + out[term_col]
    else:
        out["time_order"] = out[year_col]

    return out, year_col, term_col


df_total_t, ycol_total, tcol_total = build_time_order(df_total)
df_carrera_t, ycol_carrera, tcol_carrera = build_time_order(df_carrera)

target_total = infer_target_col(df_total_t, excluded={ycol_total, tcol_total, "time_order"})
target_carrera = infer_target_col(df_carrera_t, excluded={ycol_carrera, tcol_carrera, "time_order", "id_carrera", "carrera_id"})

print("Target total:", target_total)
print("Target carrera:", target_carrera)
print("Year col total/carrera:", ycol_total, ycol_carrera)
print("Term col total/carrera:", tcol_total, tcol_carrera)

In [ ]:
# Descripcion:
# Esta celda agrega la serie temporal total de matriculados,
# muestra estadisticas descriptivas y grafica su evolucion en el tiempo.
# Serie agregada TOTAL: evolucion temporal
series_total = (
    df_total_t.groupby("time_order", as_index=False)[target_total]
    .sum()
    .sort_values("time_order")
)

display(series_total.describe(include="all"))
display(series_total.tail(10))

plt.figure(figsize=(12, 4))
sns.lineplot(data=series_total, x="time_order", y=target_total, marker="o")
plt.title("Matriculados - Total universidad (serie temporal)")
plt.xlabel("time_order (anio*10 + cuatrimestre)")
plt.ylabel("matriculados")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Descripcion:
# Esta celda analiza la distribucion de matriculados por cuatrimestre
# en la vista total para detectar posibles efectos estacionales.
# Distribucion por cuatrimestre en TOTAL (si existe columna de cuatrimestre)
if tcol_total is not None:
    agg_term = (
        df_total_t.groupby(tcol_total, as_index=False)[target_total]
        .sum()
        .sort_values(tcol_total)
    )
    display(agg_term)

    plt.figure(figsize=(6, 4))
    sns.barplot(data=agg_term, x=tcol_total, y=target_total)
    plt.title("Matriculados acumulados por cuatrimestre (TOTAL)")
    plt.tight_layout()
    plt.show()
else:
    print("No hay columna de cuatrimestre en la vista total.")

In [ ]:
# Descripcion:
# Esta celda identifica las carreras con mayor volumen historico
# y grafica su evolucion temporal para comparar comportamientos.
# Top carreras por volumen y evolucion temporal de las principales
career_col = "id_carrera" if "id_carrera" in df_carrera_t.columns else "carrera_id"

agg_carrera = (
    df_carrera_t.groupby(career_col, as_index=False)[target_carrera]
    .sum()
    .sort_values(target_carrera, ascending=False)
)

display(agg_carrera.head(15))

top_k = 8
top_ids = agg_carrera[career_col].head(top_k).tolist()

series_top = (
    df_carrera_t[df_carrera_t[career_col].isin(top_ids)]
    .groupby(["time_order", career_col], as_index=False)[target_carrera]
    .sum()
    .sort_values([career_col, "time_order"])
)

plt.figure(figsize=(13, 5))
sns.lineplot(data=series_top, x="time_order", y=target_carrera, hue=career_col, marker="o")
plt.title(f"Evolucion temporal - Top {top_k} carreras por matriculados")
plt.xlabel("time_order (anio*10 + cuatrimestre)")
plt.ylabel("matriculados")
plt.xticks(rotation=45)
plt.legend(title=career_col, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Descripcion:
# Esta celda construye un resumen ejecutivo del EDA
# (snapshots usados, volumen, targets y cardinalidad de carreras)
# para dejar trazabilidad hacia la etapa de modelado.
# Salida de apoyo para modelado
summary = {
    "snapshot_total": key_total,
    "snapshot_carrera": key_carrera,
    "rows_total": int(df_total.shape[0]),
    "rows_carrera": int(df_carrera.shape[0]),
    "target_total": target_total,
    "target_carrera": target_carrera,
    "n_carreras": int(df_carrera_t[career_col].nunique()),
    "horizonte_total_puntos": int(series_total.shape[0]),
}

pd.DataFrame([summary])